In [111]:
import pickle
import pandas as pd
import datetime
import os

from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA
from langchain.agents import AgentExecutor, create_tool_calling_agent, tool

from ragas import evaluate, RunConfig
from ragas import EvaluationDataset
from ragas.dataset_schema import EvaluationResult
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness

In [112]:
load_dotenv()

True

In [43]:
train_df = pd.read_json("train_short.json")

# traindf where wellformedanswers is not empty
train_df.drop(columns=["query_id", "query_type", "wellFormedAnswers"], inplace=True)

# for answer in column answers apply lambda function to get the first element
train_df["answers"] = train_df["answers"].apply(lambda x: x[0] if len(x) > 0 else None)

# split passages into twenty columns (passage_0_text, passage_0_is_selected, passage_1_text, passage_1_is_selected, ...)
for i in range(10):
    passage_text = train_df["passages"].apply(lambda x: x[i]["passage_text"])
    passage_is_selected = train_df["passages"].apply(lambda x: x[i]["is_selected"])
    train_df[f"passage_{i}_selected"] = passage_is_selected
    train_df[f"passage_{i}_text"] = passage_text

# drop columns passage
train_df.drop(columns=["passages"], inplace=True)

# train_df = train_df.iloc[:10]
train_df = train_df[:200]
train_df

,answers,query,passage_0_selected,passage_0_text,passage_1_selected,passage_1_text,passage_2_selected,passage_2_text,passage_3_selected,passage_3_text,...,passage_5_selected,passage_5_text,passage_6_selected,passage_6_text,passage_7_selected,passage_7_text,passage_8_selected,passage_8_text,passage_9_selected,passage_9_text
0,The immediate impact of the success of the man...,)what was the immediate impact of the success ...,1,The presence of communication amid scientific ...,0,The Manhattan Project and its atomic bomb help...,0,Essay on The Manhattan Project - The Manhattan...,0,The Manhattan Project was the name for a proje...,...,0,The Manhattan Project. This once classified ph...,0,Nor will it attempt to substitute for the extr...,0,Manhattan Project. The Manhattan Project was a...,0,"In June 1942, the United States Army Corps of ...",0,One of the main reasons Hanford was selected a...
1,Restorative justice that fosters dialogue betw...,_________ justice is designed to repair the ha...,0,"group discussions, community boards or panels ...",0,punishment designed to repair the damage done ...,0,Tutorial: Introduction to Restorative Justice....,0,"Organize volunteer community panels, boards, o...",...,0,Each of these types of communities—the geograp...,1,The approach is based on a theory of justice t...,0,Inherent in many people’s understanding of the...,0,"Criminal justice, however, is not usually conc...",0,The circle includes a wide range of participan...
2,The customer care number of Amex India is 1800...,amex india customer care number,0,American Express is the world's premier servic...,0,Customer Service Main Page: Get Information: U...,1,Amex Card India Customer Care Phone Number Pho...,0,Air India American Express Gold Card Customer ...,...,0,"Update Address, Payee Name, Bank Account and C...",0,Please mention the first 11 digits of your Ame...,0,"The postal and official address, email address...",0,Corporate Cardmembers can contact the 24 hour ...,0,Through which any customers can easily contact...
3,"Ramen is a quick-cooking noodles, typically se...",definition of ramen,0,"Ramen is of Chinese origin, however it is uncl...",0,"ramen definition, meaning, what is ramen: a Ja...",0,Definition of ramen written for English Langua...,0,Sapporo ramen comes from Japan's northernmost ...,...,0,Wiktionary (0.00 / 0 votes) Rate this definiti...,0,Related dishes. 1 Nagasaki champon. The noodl...,0,[sometimes with sing. v.] Japanese noodles of ...,0,"‘The noodles, most of which we left behind bec...",0,"On the ground floor level, there is a souvenir..."
4,Rachel Carson died because of cancer.,why did rachel carson die,0,A Fish and Wildlife Service National Wildlife ...,0,Though much of their correspondence was destro...,0,The impetus for Silent Spring was a letter wri...,0,How about the environmentalist and writer Rach...,...,0,"Rachel Carson was born on May 27, 1907, on a f...",0,Lived 1907 – 1964. Rachel Carson played a key ...,0,Rachel Carson had a large love for nature and ...,1,Heroes commit feats of courage involving risk ...,0,"To passers-by the mother would say, pointing, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,One of the most common reasons for an HP wirel...,why does a network printer go offline,0,If It Is A Shared Network Printer. If the prin...,0,Why did your HP Wi-Fi printer go offline? A: A...,0,A: One of the most common reasons for an HP wi...,0,2) Click on Devices and Printers in Large or S...,...,0,"However, if this option does not solve your pr...",1,Quick Answer. One of the most common reasons f...,0,The following settings need to be checked; 1) ...,0,Why is My Printer Offline? there could be seve...,0,Let’s Fix it. 1) Go to control panel. On Windo...
196,A person waives pip coverage because they beli...,why does a person waives pip coverage,0,"In such states PIP coverage is not mandated, g...",0,"Alternatively, if your health insurance doesn'...",0,Medical 

,answers,query,passage_0_selected,passage_0_text,passage_1_selected,passage_1_text,passage_2_selected,passage_2_text,passage_3_selected,passage_3_text,...,passage_5_selected,passage_5_text,passage_6_selected,passage_6_text,passage_7_selected,passage_7_text,passage_8_selected,passage_8_text,passage_9_selected,passage_9_text
34,Britain and France declared war on Germany fol...,why did we declare war on germany?,0,"For example: 1 In 1905 and 1911, there were d...",0,The United States later declared war on German...,1,Why did Britain and France declare war on Germ...,0,"On 11 December 1941, four days after the Japan...",...,0,54 Answers. The United States entered the war ...,0,Why did the U.S. declare war on Germany during...,0,The President said a declaration of war would ...,0,We will not only defend ourselves to the utter...,0,"On 11 December 1941, four days after the Japan..."


In [44]:
sample_queries = list(train_df['query'])
expected_responses = list(train_df['answers'])

# create a list of every passage, also append the rowid stretched to 4 digits + passage_0_selected number
passages = []
for i, row in train_df.iterrows():
    for j in range(10):
        passage = row[f"passage_{j}_text"]
        passage_is_selected = row[f"passage_{j}_selected"]
        id = f"{int(i):04d}_{passage_is_selected}"
        passages.append((id, passage))

In [45]:
dataset_documents = [
    Document(
        page_content=one_passage,
        metadata={
            "id": one_id,
        }
    ) for one_id, one_passage in passages
]

In [46]:
embeddings = OllamaEmbeddings(model="mxbai-embed-large")
dataset_vector_store = InMemoryVectorStore(embeddings)

In [47]:
dataset_vector_store.add_documents(dataset_documents)

['9d0632ec-8d82-4c6f-b9c6-97cd205bde4c',
 '9de3d825-e817-444e-8771-78edf666d536',
 '831cdd09-50fe-4c2d-a991-b8a344f47f69',
 '9f9c1bc2-4247-472d-a73c-36581e7f25d4',
 'ab09098c-ab3d-441f-b0f1-777789672a47',
 'e09c47e6-de50-4d05-9ff8-31a48d67c0b0',
 '49889dbf-fd00-457e-b73a-537f812dfbaa',
 'c5d3b565-2c90-4e63-a59d-4b9cb3482418',
 '52c920f3-bada-4e0e-883a-c34a6337c655',
 '54653dce-dfd9-4759-8175-9a80644542f9',
 '41837272-2e76-4be9-a6f4-95a9d51c8f95',
 'bd37adac-598a-4632-981a-3f1c9ee78c85',
 '1fd8fae3-e8c2-41e5-9e05-d6655afb2597',
 'c289d7af-7ccd-441d-a2ad-ace883de416a',
 '179c503f-cb30-437e-bb54-321bd4348eee',
 '8de9a6ff-3159-4d58-bba7-f49bee59c0c4',
 '5b87c404-49aa-4896-8caf-37438cb5575c',
 '144b8eff-3900-4ed7-a4df-7af731991bcc',
 '944212c5-6994-4cdb-ac3d-393fed72a7f9',
 '66ce716c-8e21-478b-a105-3c5ca6a94bf6',
 '69fa4c64-1b3a-4ad0-86e8-e3c7e12a7423',
 '6f878a2f-06d4-4b04-af6e-f436e225e372',
 'f6ce248d-38d3-458a-b7e6-17e16057b9d7',
 'efd0fcaa-3fb5-4ffc-a2b0-7619d9fd4317',
 'db2fa254-9d82-

In [113]:
GOOGLE_KEYS = []

for i in range(10):
    possible_name = f"GOOGLE_API_KEY_{i + 1}"
    if os.getenv(possible_name):
        GOOGLE_KEYS.append(os.getenv(possible_name))

key_index = 0
def get_next_google_key():
    global key_index
    key = GOOGLE_KEYS[key_index]
    key_index = (key_index + 1) % len(GOOGLE_KEYS)
    return key

In [90]:
system_prompt = """
You are a helpful reasearch assistant. Your task is to help the user with their research and do the requested tasks no matter how difficult they are.
You should always try to use the rag tools to check if the answer can be found in the database of text passages. Try very hard to find the answer in the database.
You are given access to a database of text passages and a RAG system that can help you access the information about the selected passages.
When you are asked a question, you should first check if the answer is in the database.
If the question is not related to the passages, inform the user that the question is not related to the passages (it will happen rarely) and you cannot help them with that (BUT ONLY AFTER YOU HAVE CHECKED IT USING THE TOOLS!).
If you are unsure whether it is related to the passages, you should assume that it IS. You MUST always assume at first that the question is related to the passages and check it using the tools.
You should use tools rather too much than too little. You should try your best to answer the question using the tools.
You can use the tools as many times as you want, prioritirize using elastic_search_tool over rag_tool, but if you cannot find the answer using elastic_search_tool, you should use rag_tool.
REMEMBER: the passages are about a lot of different topics, so you should not assume that the question is not related to the passages just because it is about a specific topic.
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

In [ ]:
def answer_question(question: str) -> tuple[str, list[str]]:
    os.environ["GOOGLE_API_KEY"] = get_next_google_key()

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash-preview-04-17",
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=1,
    )

    rag_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=dataset_vector_store.as_retriever(search_kwargs={"k": 3}),
    )

    retrieved_context = []

    @tool
    def rag_tool(query: str) -> str:
        """Use this tool to get information from the RAG model if needed.
        This tool allows you to ask questions about the papers selected by the user.
        This tool should return the answer with all the necessary citations.
        Sometimes the tool will not be able to answer the question, in that case, you should try to formulate the question in a different way.
        Most often than not, the problem with this tool's answer will be in your question formulation, so you should try to rephrase it them
        as this tool will try to answer your question very literally.
        """
        output = rag_chain.invoke(query)
        retrieved_context.append(output['result'])
        return output

    @tool
    def elastic_search_tool(query: str) -> str:
        """Use this tool to get information from the Elasticsearch index if needed.
        This tool allows you to ask questions about the papers selected by the user.
        This tool should return the answer together with the metadata of the paper, by default returning the first 3 results.
        """
        results = dataset_vector_store.similarity_search(query, k=3)

        response = ""
        for i, result in enumerate(results):
            response += f"Result {i+1}:\n"
            response += f"ID: {result.metadata['id']}\n" if "id" in result.metadata else "N/A\n"
            # response += f"Title: {result.metadata['title']}\n" if "title" in result.metadata else "N/A"
            # response += f"Authors: {result.metadata['authors']}\n" if "authors" in result.metadata else "N/A"
            # response += f"DOI: {result.metadata['doi']}\n" if "doi" in result.metadata else "N/A"
            # response += f"Page: {result.metadata['page']}\n\n" if "page" in result.metadata else "N/A\n\n"
            response += f"Content:\n{result.page_content}\n\n\n\n"

        retrieved_context.append(response)
        return response

    tools = [rag_tool, elastic_search_tool]

    agent = create_tool_calling_agent(llm, tools, prompt)
    agent_executor = AgentExecutor(
        agent=agent, tools=tools, verbose=True, max_iterations=15
    )
    output = agent_executor.invoke({"input": question})[
        "output"
    ]

    print(f"Question: {question}")
    print(f"Answer: {output}")

    return output, retrieved_context

In [88]:
def build_evaluation_dataset(queries, references, retriever, formatter):
    data = []
    for query, reference in zip(queries, references):
        response, relevant_docs = answer_question(query)
        data.append({
            "user_input": query,
            "retrieved_contexts": relevant_docs, # [f"Document id {doc.metadata["id"]}:\n{doc.page_content}" for doc in relevant_docs],
            "response": response,
            "reference": reference,
        })
    return EvaluationDataset.from_list(data)


def format_docs(relevant_docs):
    return [f"Document:\n{doc.page_content}\n\n" for doc in relevant_docs]

In [ ]:
evaluation_dataset = build_evaluation_dataset(
    list(train_df['query']),
    list(train_df['answers']),
    dataset_vector_store,
    format_docs
)



> Entering new AgentExecutor chain...

Invoking: `elastic_search_tool` with `{'query': 'why did we declare war on germany?'}`


Result 1:
ID: 0034_0
Content:
Why did the U.S. declare war on Germany during WW2? World War II: After the U.S. declared war on Japan following their surprise attack on Pearl Harbor, it was clear that the U.S. could not stay out of World War II as a neutral party. However, despite the alliance Japan had with Germany and the U.S. had with Britain, it was not clear that the U.S. had a national interest in war with Germany nor that Germany had one with the U.S. Why did the U.S. declare war on Germany?



Result 2:
ID: 0034_1
Content:
Why did Britain and France declare war on Germany in September 1939? Britain and France declared war on Germany following the invasion of Poland two days before. This was due to many factors, including the famous Nazi-Soviet Pact; where Russia and Germany split Poland in half, resultantly giving Germany the confidence to invade Pola

In [110]:
datetime_str = datetime.datetime.now().strftime("%Y_%m_%d_%H_%M_%S")

with open(f"evaluation_dataset_{datetime_str}.pickle", "wb") as f:
    pickle.dump(evaluation_dataset, f)

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-preview-04-17",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=1,
)

In [99]:
run_config = RunConfig(timeout=60*10, max_workers=1)
evaluator_llm = LangchainLLMWrapper(llm)

In [100]:
result_correctness: EvaluationResult = evaluate(
    dataset=evaluation_dataset,
    metrics=[FactualCorrectness()],
    llm=evaluator_llm,
    run_config=run_config,
)

result_correctness

Evaluating: 100%|██████████| 1/1 [00:23<00:00, 23.89s/it]


{'factual_correctness(mode=f1)': 0.4000}